# REFRESH — Extractive Summarization với Reinforcement Learning
**Môn CS106 — Trí tuệ nhân tạo, UIT**

Paper: *Ranking Sentences for Extractive Summarization with Reinforcement Learning* (Narayan et al., NAACL 2018)

Code gốc: https://github.com/EdinburghNLP/Refresh (TF 0.10, Python 2)

Port: PyTorch hiện đại, chạy trên CNN/DailyMail subset trên Colab GPU.

> **Ban danh cho App demo** -- chinh tu `REFRESH_experiment_after_run_GloVe.ipynb`:
> 1. **Eval tren FULL test split (11,490 docs)** thay vi 2,000 (`N_TEST=None`).
> 2. **ROUGE faithful theo paper**: `rougeLsum` + Porter stemmer (`FAITHFUL_ROUGE=True`) -> ROUGE-L khop paper (~36) thay vi ~22.
> 3. **MAX_DOC_LEN=120** (dung paper).
> 4. Baseline doi ten **"LEAD"** (dung paper, khong phai "LEAD-3").
> 5. Them cell **EXPORT FOR APP** o cuoi de xuat checkpoint cho Streamlit.
>
> WARN: So ROUGE se KHAC slide cu (do test 11,490 + metric faithful). Cap nhat bang slide tu output moi.


In [ ]:
# Cài đặt dependencies
!pip install datasets rouge-score nltk tqdm matplotlib seaborn gensim -q


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random, os, itertools, time, pickle
from multiprocessing import Pool
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Config — giữ NGUYÊN paper hyperparams, chỉ scale-down TRAIN ──────────
# Dataset gốc CNN/DailyMail: 287k/13k/11k → train quá lớn cho Colab 12h session.
# Train 50k (scale-up từ 15k để giảm overfit), Val 2k.
# TEST = FULL 11,490 docs (N_TEST=None) → eval đúng cỡ test của paper.
# Mọi tham số khác giữ NGUYÊN paper Section 5.
N_TRAIN, N_VAL            = 50000, 2000
N_TEST                    = None   # None = FULL test split (11,490 docs) -- eval DUNG theo paper
EMB_DIM,  LSTM_HID        = 200,   600          # paper Section 5 (my_flags.py:41,75)
MAX_DOC_LEN, MAX_SENT_LEN = 120,   100          # paper Section 5: max doc length 120, sent 100
EPOCHS_RL, BATCH_SIZE     = 10,    32           # paper: 20 ep × 20; mình 10 × 32 cho tốc độ
LR                        = 1e-3                # paper Section 5 (my_flags.py:104) — 50k đủ lớn để dùng LR paper
GRAD_CLIP                 = 5.0                 # code gốc model_docsum.py:790
VOCAB_SIZE                = 50000               # subset 50k cần vocab ~50k (paper KHÔNG nêu vocab size)
USE_AMP                   = torch.cuda.is_available()  # mixed precision ~2x speed trên GPU

SENT_EMB    = 350           # paper: concat 7 kernels × 50 ch (my_flags.py:50)
NUM_KERNELS = 7             # my_flags.py:64
assert SENT_EMB % NUM_KERNELS == 0

m, p, k = 3, 10, 5          # paper CNN: m=3,p=10,k=5 (DailyMail dùng m=4,k=15 — đây dùng 1 config chung)

# -- ROUGE faithful theo paper (pyrouge '-a -c 95 -m -n 4 -w 1.2') ---------------
#   rougeLsum (summary-level union-LCS) thay vi rougeL (sentence-level) -> RL khop paper
#   use_stemmer=True  (co '-m' = Porter stemming)
# Dat False de tai lap dung so CU tren slide (rougeL + no-stemmer, RL ~22).
FAITHFUL_ROUGE = True

# Local checkpoint dir (subset đủ nhỏ — không cần mount Drive)
SAVE_DIR = './refresh_ckpt'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Train/Val = {N_TRAIN}/{N_VAL} | Test = FULL split' if N_TEST is None else f'Train/Val/Test = {N_TRAIN}/{N_VAL}/{N_TEST}')
print(f'EMB={EMB_DIM}, LSTM_HID={LSTM_HID}, m={m}, p={p}, k={k}, LR={LR}, AMP={USE_AMP}')
print(f'Checkpoints: {SAVE_DIR}')


In [ ]:
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from datasets import load_dataset
from collections import Counter

print('Loading CNN/DailyMail...')
# Bản datasets/huggingface_hub mới bắt buộc repo id dạng 'namespace/name'.
# Tên cũ 'cnn_dailymail' không còn hợp lệ → dùng mirror 'abisee/cnn_dailymail'.
raw = load_dataset('abisee/cnn_dailymail', '3.0.0')

def tokenize_doc(text, max_sents, max_words):
    # Lowercase: paper KHÔNG lowercase, nhưng GloVe wiki-gigaword là uncased -> phải lower để có coverage.
    sents = sent_tokenize(text)[:max_sents]
    return [word_tokenize(s.lower())[:max_words] for s in sents]

def tokenize_highlights(text):
    return word_tokenize(text.lower())[:200]

train_raw = list(raw['train'])[:N_TRAIN]
val_raw   = list(raw['validation'])[:N_VAL]
test_raw  = list(raw['test']) if N_TEST is None else list(raw['test'])[:N_TEST]

# Build vocab from train
counter = Counter()
for item in tqdm(train_raw, desc='Vocab'):
    for s in tokenize_doc(item['article'], MAX_DOC_LEN, MAX_SENT_LEN):
        counter.update(s)

PAD_ID, UNK_ID = 0, 1
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
word2id = {w: i for i, w in enumerate(vocab)}
ACTUAL_VOCAB_SIZE = len(vocab)
print(f'Vocab: {ACTUAL_VOCAB_SIZE}')

def encode_doc(text):
    sents = tokenize_doc(text, MAX_DOC_LEN, MAX_SENT_LEN)
    return [[word2id.get(w, UNK_ID) for w in s] + [PAD_ID] * (MAX_SENT_LEN - len(s)) for s in sents]

def process_split(raw_split, desc):
    data = []
    for item in tqdm(raw_split, desc=desc):
        doc = encode_doc(item['article'])
        if len(doc) < 2:
            continue
        data.append({
            'doc': doc,
            'raw_sents': tokenize_doc(item['article'], MAX_DOC_LEN, MAX_SENT_LEN),
            'raw_gold': item['highlights'],
        })
    return data

train_data = process_split(train_raw, 'Train')
val_data   = process_split(val_raw,   'Val')
test_data  = process_split(test_raw,  'Test')
print(f'Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}')


## Pretrained Word Embeddings — GloVe 200d
> Paper Section 5: skip-gram trên One Billion Word Benchmark (~1B từ tin tức).
> Mình dùng **GloVe 200d** (Wikipedia+Gigaword, ~252MB) thay thế vì không train được skip-gram OBW trên Colab.
> Khác algorithm (GloVe = co-occurrence matrix factorization vs skip-gram = predict context),
> nhưng cùng spirit: pretrained distributional word embeddings 200d.
> OOV words init zero theo paper.

In [ ]:
import gensim.downloader as api

# Download + load GloVe 200d (Wikipedia + Gigaword 6B tokens, ~252MB)
# Lần đầu: ~1-2 phút download. Lần sau: load từ cache ~/gensim-data/.
print('Loading GloVe 200d (glove-wiki-gigaword-200)...')
glove = api.load('glove-wiki-gigaword-200')
print(f'  → {len(glove.key_to_index):,} từ, dim={glove.vector_size}')
assert glove.vector_size == EMB_DIM, f'GloVe dim {glove.vector_size} ≠ EMB_DIM {EMB_DIM}'

# Build embedding matrix [vocab_size, EMB_DIM] cho model
# Paper Section 5: known words = pretrained, unknown words = zero init.
embedding_matrix = np.zeros((ACTUAL_VOCAB_SIZE, EMB_DIM), dtype=np.float32)
hits = 0
for word, idx in word2id.items():
    if word in glove.key_to_index:
        embedding_matrix[idx] = glove[word]
        hits += 1
    # else: giữ zero (paper: "Embeddings for unknown words were initialized to zero")
embedding_matrix[PAD_ID] = 0.0  # PAD luôn zero

print(f'GloVe coverage: {hits:,}/{ACTUAL_VOCAB_SIZE:,} ({hits/ACTUAL_VOCAB_SIZE*100:.1f}%)')
print(f'OOV (zero-init, train end-to-end): {ACTUAL_VOCAB_SIZE - hits:,}')

del glove  # free ~640MB RAM


## Oracle Ŷ — Top-k High-ROUGE Summaries (Paper Section 4.2)
> Top-p=10 câu individual → C(10,m) combinations → giữ top-k theo mean(R1,R2,RL) F1

In [ ]:
from rouge_score import rouge_scorer

# FAITHFUL_ROUGE (cell config): rougeLsum + stemmer khop pyrouge cua paper.
_RL_KEY = 'rougeLsum' if FAITHFUL_ROUGE else 'rougeL'
_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', _RL_KEY], use_stemmer=FAITHFUL_ROUGE)

def _join_sents(sents_tokens):
    # Ghep list[list[token]] -> string. rougeLsum can newline giua cac cau (union-LCS).
    sep = chr(10) if FAITHFUL_ROUGE else ' '
    return sep.join(' '.join(w) for w in sents_tokens)

def rouge_f(candidate_text, reference_text):
    s = _scorer.score(reference_text, candidate_text)
    return (s['rouge1'].fmeasure + s['rouge2'].fmeasure + s[_RL_KEY].fmeasure) / 3.0

def compute_oracle_for_doc(item, m, p, k):
    sents, gold = item['raw_sents'], item['raw_gold']
    n = len(sents)
    if n < m:
        return [(tuple(range(n)), 0.0)]

    sent_scores = sorted(
        [(rouge_f(' '.join(s), gold), i) for i, s in enumerate(sents)],
        reverse=True
    )
    top_p_ids = sorted(idx for _, idx in sent_scores[:p])

    combo_scores = sorted(
        [(rouge_f(_join_sents([sents[i] for i in combo]), gold), combo)
         for combo in itertools.combinations(top_p_ids, m)],
        reverse=True
    )
    return [(combo, score) for score, combo in combo_scores[:k]]

def _oracle_worker(args):
    return compute_oracle_for_doc(*args)

# ── Cache oracle ra disk (skip nếu restart kernel) ──
ORACLE_CACHE = os.path.join(SAVE_DIR, f'oracles_N{len(train_data)}_m{m}_p{p}_k{k}_' + ('faithful' if FAITHFUL_ROUGE else 'raw') + '.pkl')

if os.path.exists(ORACLE_CACHE):
    print(f'Loading cached oracle: {ORACLE_CACHE}')
    with open(ORACLE_CACHE, 'rb') as f:
        train_oracles = pickle.load(f)
else:
    print(f'Computing oracle Ŷ ({len(train_data)} docs, m={m}, p={p}, k={k})...')
    t0 = time.time()
    with Pool(processes=os.cpu_count()) as pool:
        train_oracles = list(tqdm(
            pool.imap(_oracle_worker, [(item, m, p, k) for item in train_data], chunksize=50),
            total=len(train_data), desc='Oracle'
        ))
    print(f'Oracle done in {(time.time()-t0)/60:.1f} min — saving cache...')
    with open(ORACLE_CACHE, 'wb') as f:
        pickle.dump(train_oracles, f)
    print(f'Saved: {ORACLE_CACHE}')

for item, oracle in zip(train_data, train_oracles):
    item['oracles'] = oracle

avg_r = np.mean([oracle[0][1] for oracle in train_oracles if oracle])
print(f'Avg best oracle ROUGE: {avg_r:.4f}')


In [ ]:
class SentenceEncoderCNN(nn.Module):
    """CNN sentence encoder — paper Section 5 + model_utils.py:111-171.
    Kernel widths 1-7, channel = sent_emb // 7 = 50, ReLU + max-pool over time,
    concat 7 outputs → sent_emb-dim vector per sentence.
    (Bỏ Local Response Normalization của code gốc — modern PyTorch hiếm dùng.)
    """
    def __init__(self, emb_dim, sent_emb, num_kernels):
        super().__init__()
        ch = sent_emb // num_kernels
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, ch, kernel_size=w) for w in range(1, num_kernels + 1)
        ])

    def forward(self, x):
        # x: [B*S, T, emb_dim] → transpose for Conv1d → [B*S, emb_dim, T]
        x = x.transpose(1, 2)
        return torch.cat([F.relu(c(x)).max(dim=2).values for c in self.convs], dim=1)


class REFRESHModel(nn.Module):
    """REFRESH — Figure 1 of paper, faithful port của model_docsum.py:165-201.

    Kiến trúc:
      1. CNN encode mỗi câu → sent_reps [B, S, 350]
      2. Document encoder LSTM đọc sent_reps theo thứ tự ĐẢO NGƯỢC
         → giữ FINAL hidden state (h_doc, c_doc) làm summary của doc
         → BỎ outputs (không dùng)
      3. Sentence extractor LSTM:
         - input  = sent_reps GỐC (350-dim, KHÔNG phải doc encoder outputs)
         - init   = (h_doc, c_doc) từ doc encoder
         → output mỗi câu → Linear(2) → logits [extract, skip]

    Tại sao reverse ở doc encoder: paper Section 2 — đảo thứ tự giúp final
    hidden state ưu tiên thông tin từ câu đầu (quan trọng nhất cho summarization).

    Tại sao extractor lấy sent_reps gốc + doc state làm init: doc encoder làm
    "context vector", extractor LSTM stream qua từng câu để label sequentially.
    Đây là pattern encoder-decoder cổ điển (Sutskever 2014), không phải stacked
    LSTM. Code gốc: sentence_extractor_nonseqrnn_noatt(sents_ext, encoder_state).

    pretrained_emb: nếu cung cấp [vocab_size, emb_dim], init embedding layer
    bằng pretrained (paper: skip-gram OBW; mình: GloVe 200d Wiki+Gigaword).
    """
    def __init__(self, vocab_size, emb_dim, sent_emb, lstm_hid, num_kernels,
                 pad_id=0, pretrained_emb=None):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        if pretrained_emb is not None:
            assert pretrained_emb.shape == (vocab_size, emb_dim), \
                f'pretrained_emb shape {pretrained_emb.shape} ≠ ({vocab_size}, {emb_dim})'
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_emb))
        self.sent_enc  = SentenceEncoderCNN(emb_dim, sent_emb, num_kernels)
        # Doc encoder: input = sent_reps reversed (350-dim)
        self.doc_lstm  = nn.LSTM(sent_emb, lstm_hid, batch_first=True)
        # Extractor: input = sent_reps gốc (350-dim, KHÔNG phải doc_out 600-dim)
        # init state sẽ = final state của doc_lstm trong forward()
        self.ext_lstm  = nn.LSTM(sent_emb, lstm_hid, batch_first=True)
        self.classifier = nn.Linear(lstm_hid, 2)

    def forward(self, docs):
        # docs: [B, S, T]
        B, S, T = docs.shape
        mask = (docs != self.pad_id).any(dim=-1)           # [B, S]

        x = self.embedding(docs.view(B * S, T))            # [B*S, T, emb_dim]
        s = self.sent_enc(x).view(B, S, -1)                # [B, S, sent_emb=350]

        # Document encoder: REVERSED sent_reps → keep FINAL state only
        _, (h_doc, c_doc) = self.doc_lstm(torch.flip(s, [1]))
        # h_doc, c_doc: [1, B, lstm_hid]

        # Sentence extractor: input = sent_reps GỐC (original order),
        #   initial state = (h_doc, c_doc) từ doc encoder
        ext_out, _ = self.ext_lstm(s, (h_doc, c_doc))      # [B, S, lstm_hid]

        logits = self.classifier(ext_out)                  # [B, S, 2]
        return logits, mask


model = REFRESHModel(
    vocab_size=ACTUAL_VOCAB_SIZE, emb_dim=EMB_DIM,
    sent_emb=SENT_EMB, lstm_hid=LSTM_HID, num_kernels=NUM_KERNELS,
    pretrained_emb=embedding_matrix,   # ← GloVe 200d init
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {n_params:,}')
print(f'Architecture (faithful paper Figure 1):')
print(f'  [B,S,T] → embed (GloVe init) → CNN → sent_reps [B,S,{SENT_EMB}]')
print(f'  sent_reps ──flip──> doc_LSTM → final (h,c)   (discard outputs)')
print(f'  sent_reps ─────────> ext_LSTM(init=(h,c)) → [B,S,{LSTM_HID}] → Linear → [B,S,2]')


In [ ]:
def lead_summary(raw_sents, m):
    return raw_sents[:m]

def evaluate_rouge(pred_sents_list, gold_texts):
    r1, r2, rl = [], [], []
    for sents, gold in zip(pred_sents_list, gold_texts):
        s = _scorer.score(gold, _join_sents(sents))
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s[_RL_KEY].fmeasure)
    return {'ROUGE-1': np.mean(r1)*100, 'ROUGE-2': np.mean(r2)*100, 'ROUGE-L': np.mean(rl)*100}

lead_scores = evaluate_rouge(
    [lead_summary(item['raw_sents'], m) for item in test_data],
    [item['raw_gold'] for item in test_data]
)
print('LEAD:', {kk: f'{v:.2f}' for kk, v in lead_scores.items()})


## Training REFRESH — Reward-Weighted CE (port từ `model_docsum.py:474`)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class REFRESHDataset(Dataset):
    def __init__(self, data, has_oracle=False):
        self.data = data
        self.has_oracle = has_oracle

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        doc_tensor = torch.zeros(MAX_DOC_LEN, MAX_SENT_LEN, dtype=torch.long)
        weight = torch.zeros(MAX_DOC_LEN)
        for i, sent in enumerate(item['doc'][:MAX_DOC_LEN]):
            doc_tensor[i, :len(sent)] = torch.tensor(sent[:MAX_SENT_LEN], dtype=torch.long)
            weight[i] = 1.0

        result = {'doc': doc_tensor, 'weight': weight}

        if self.has_oracle:
            # Random pick 1 oracle per step (data_utils.py:190)
            oracle_indices, reward = item['oracles'][random.randint(0, len(item['oracles']) - 1)]
            label = torch.ones(MAX_DOC_LEN, dtype=torch.long)  # default: skip (1)
            for oi in oracle_indices:
                if oi < MAX_DOC_LEN:
                    label[oi] = 0  # extract (0)
            result['oracle_label'] = label
            result['reward'] = torch.tensor(reward, dtype=torch.float)

        return result

train_loader = DataLoader(REFRESHDataset(train_data, has_oracle=True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(REFRESHDataset(val_data),                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(REFRESHDataset(test_data),                   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')


In [ ]:
from torch.cuda.amp import autocast, GradScaler

def reward_weighted_ce_loss(logits, oracle_label, reward, weight):
    """Port từ reward_weighted_cross_entropy_loss_multisample (model_docsum.py:474).
    CE per token × padding mask × scalar reward → mean over batch."""
    B, S, _ = logits.shape
    ce = F.cross_entropy(logits.view(B*S, 2), oracle_label.view(B*S), reduction='none')
    ce = ce.view(B, S) * weight
    return (ce.sum(dim=1) * reward).mean()

def predict_topm(logits, weight, m):
    """Test inference: top-m câu theo p(extract=0), tie-break theo index nhỏ trước
    (predict_topranked, model_utils.py:301)."""
    probs = F.softmax(logits, dim=-1)[:, :, 0] * weight
    topk  = torch.topk(probs, k=min(m, probs.size(1)), dim=1)
    return [sorted(topk.indices[b].tolist()[:m]) for b in range(logits.size(0))]

def eval_rouge(model, data_list, loader, m):
    model.eval()
    preds, golds, idx = [], [], 0
    with torch.no_grad():
        for batch in loader:
            with autocast(enabled=USE_AMP):
                logits, _ = model(batch['doc'].to(device))
            for b, sent_ids in enumerate(predict_topm(logits.float(), batch['weight'].to(device), m)):
                item = data_list[idx + b]
                preds.append([item['raw_sents'][i] for i in sent_ids if i < len(item['raw_sents'])])
                golds.append(item['raw_gold'])
            idx += logits.size(0)
    return evaluate_rouge(preds, golds)


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler    = GradScaler(enabled=USE_AMP)

BEST_CKPT = os.path.join(SAVE_DIR, 'refresh_best.pt')
LAST_CKPT = os.path.join(SAVE_DIR, 'refresh_last.pt')

# ── Force fresh start nếu checkpoint cũ không tương thích config hiện tại ──
# Check 3 thứ:
#   (1) LR mismatch                       → optimizer momentum sẽ sai
#   (2) embedding init mismatch           → ckpt random init nhưng config GloVe (hoặc ngược lại)
#   (3) vocab_size mismatch               → embedding.weight shape khác → load_state_dict fail
# Đảm bảo rerun với config mới không bị "load đè" weight cũ.
CFG_HAS_PRETRAINED_EMB = True   # config hiện tại dùng GloVe (Cell 5)
CFG_VOCAB_SIZE         = ACTUAL_VOCAB_SIZE
FORCE_FRESH = False
if os.path.exists(LAST_CKPT):
    try:
        ck_peek = torch.load(LAST_CKPT, map_location='cpu', weights_only=False)  # ckpt tự sinh (có optim state)
        ckpt_lr     = ck_peek['optim']['param_groups'][0]['lr']
        ckpt_emb    = ck_peek.get('has_pretrained_emb', False)  # ckpt cũ không có flag = random init
        ckpt_vocab  = ck_peek.get('vocab_size', None)            # ckpt cũ không có flag = None
        if abs(ckpt_lr - LR) > 1e-9:
            print(f'⚠️  LR trong checkpoint ({ckpt_lr}) ≠ LR config ({LR}) → fresh start.')
            FORCE_FRESH = True
        elif ckpt_emb != CFG_HAS_PRETRAINED_EMB:
            print(f'⚠️  Checkpoint cũ has_pretrained_emb={ckpt_emb} nhưng config={CFG_HAS_PRETRAINED_EMB} → fresh start.')
            FORCE_FRESH = True
        elif ckpt_vocab is None or ckpt_vocab != CFG_VOCAB_SIZE:
            print(f'⚠️  Vocab size trong checkpoint ({ckpt_vocab}) ≠ config ({CFG_VOCAB_SIZE}) → fresh start.')
            FORCE_FRESH = True
        del ck_peek
    except Exception as e:
        print(f'⚠️  Không đọc được checkpoint cũ ({e}) → fresh start.')
        FORCE_FRESH = True

if FORCE_FRESH:
    for f in (LAST_CKPT, BEST_CKPT):
        if os.path.exists(f):
            os.remove(f)

# ── Resume từ checkpoint nếu tồn tại (và không bị force fresh) ──
start_epoch       = 1
train_losses      = []
val_rouge_history = []
best_val_r1       = -1.0
best_epoch        = 0

if os.path.exists(LAST_CKPT):
    try:
        print(f'Resume từ {LAST_CKPT}...')
        ck = torch.load(LAST_CKPT, map_location=device, weights_only=False)  # torch>=2.6: cần để resume được
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optim'])
        if 'scaler' in ck and USE_AMP and ck['scaler'] is not None:
            scaler.load_state_dict(ck['scaler'])
        start_epoch       = ck['epoch'] + 1
        train_losses      = ck.get('train_losses', [])
        val_rouge_history = ck.get('val_rouge_history', [])
        best_val_r1       = ck.get('best_val_r1', -1.0)
        best_epoch        = ck.get('best_epoch', 0)
        print(f'  → resuming from epoch {start_epoch}, best val R1 so far = {best_val_r1:.2f}')
    except (RuntimeError, KeyError) as e:
        # Checkpoint cũ không tương thích (vd: code architecture đã đổi) → xóa, train lại
        print(f'⚠️  Checkpoint cũ không tương thích — xóa và train lại từ đầu.\n   Error: {e}')
        for f in (LAST_CKPT, BEST_CKPT):
            if os.path.exists(f):
                os.remove(f)
        # Reset state (model + optimizer + scaler vẫn fresh từ đầu cell)
        start_epoch = 1
        train_losses, val_rouge_history = [], []
        best_val_r1, best_epoch = -1.0, 0

print(f'\nTraining REFRESH: {EPOCHS_RL} epochs × {len(train_loader)} batches | LR={LR} | AMP={USE_AMP} | batch={BATCH_SIZE}')
t_start = time.time()

# Outer tqdm cho ETA tổng thể
epoch_bar = tqdm(range(start_epoch, EPOCHS_RL + 1), desc='Epochs', position=0)
for epoch in epoch_bar:
    model.train()
    losses = []
    inner_bar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS_RL}', leave=False, position=1)
    for batch in inner_bar:
        docs   = batch['doc'].to(device, non_blocking=True)
        weight = batch['weight'].to(device, non_blocking=True)
        label  = batch['oracle_label'].to(device, non_blocking=True)
        reward = batch['reward'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            logits, _ = model(docs)
            loss = reward_weighted_ce_loss(logits, label, reward, weight)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        losses.append(loss.item())
        inner_bar.set_postfix(loss=f'{loss.item():.3f}')

    avg_loss = np.mean(losses)
    train_losses.append(avg_loss)
    val_s = eval_rouge(model, val_data, val_loader, m)
    val_rouge_history.append(val_s)

    # Save BEST checkpoint (theo val ROUGE-1)
    marker = ''
    if val_s['ROUGE-1'] > best_val_r1:
        best_val_r1 = val_s['ROUGE-1']
        best_epoch  = epoch
        torch.save(model.state_dict(), BEST_CKPT)
        marker = '  ★ best'

    # Save LAST (resume state) mỗi epoch
    torch.save({
        'epoch': epoch, 'model': model.state_dict(), 'optim': optimizer.state_dict(),
        'scaler': scaler.state_dict() if USE_AMP else None,
        'train_losses': train_losses, 'val_rouge_history': val_rouge_history,
        'best_val_r1': best_val_r1, 'best_epoch': best_epoch,
        'has_pretrained_emb': CFG_HAS_PRETRAINED_EMB,   # ← cho safe-guard lần sau
        'vocab_size':         CFG_VOCAB_SIZE,           # ← cho safe-guard lần sau
    }, LAST_CKPT)

    # Per-epoch ETA
    elapsed   = time.time() - t_start
    done      = epoch - start_epoch + 1
    remaining = EPOCHS_RL - epoch
    eta_sec   = (elapsed / done) * remaining if done > 0 else 0

    epoch_bar.set_postfix({
        'loss': f'{avg_loss:.3f}', 'val_R1': f'{val_s["ROUGE-1"]:.2f}',
        'best': f'{best_val_r1:.2f}@ep{best_epoch}',
        'elapsed': f'{elapsed/60:.1f}m', 'ETA': f'{eta_sec/60:.1f}m',
    })
    print(f'Epoch {epoch}: loss={avg_loss:.4f} | Val R1={val_s["ROUGE-1"]:.2f} R2={val_s["ROUGE-2"]:.2f} RL={val_s["ROUGE-L"]:.2f} | elapsed={elapsed/60:.1f}m ETA={eta_sec/60:.1f}m{marker}')

# Load BEST checkpoint cho eval/chart cells downstream
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
print(f'\n✓ Done in {(time.time()-t_start)/60:.1f} min | Best: epoch {best_epoch} | Val R1={best_val_r1:.2f}')
print(f'  Best ckpt:  {BEST_CKPT}')
print(f'  Last ckpt:  {LAST_CKPT}')


In [ ]:
# ── OPTIONAL: CE warm-start (chỉ dùng nếu training cell trên không converge sau 3 epoch) ──
# Paper Section 4.2: MIXER (CE+RL) "performed worse" → không dùng mặc định.
# Nếu cần: đổi RUN_CE_WARMUP = True

# RUN_CE_WARMUP = False
# if RUN_CE_WARMUP:
#     model_ce = REFRESHModel(
#         vocab_size=ACTUAL_VOCAB_SIZE, emb_dim=EMB_DIM,
#         sent_emb=SENT_EMB, lstm_hid=LSTM_HID, num_kernels=NUM_KERNELS,
#     ).to(device)
#     opt_ce = torch.optim.Adam(model_ce.parameters(), lr=LR)
#
#     for epoch in range(1, 3):  # 2 epochs CE warm-start
#         model_ce.train()
#         for batch in tqdm(train_loader, desc=f'CE Warmup {epoch}'):
#             docs, weight = batch['doc'].to(device), batch['weight'].to(device)
#             label = batch['oracle_label'].to(device)
#             opt_ce.zero_grad()
#             logits, _ = model_ce(docs)
#             B, S, _ = logits.shape
#             ce = F.cross_entropy(logits.view(B*S, 2), label.view(B*S), reduction='none')
#             loss = (ce.view(B, S) * weight).mean()
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model_ce.parameters(), GRAD_CLIP)
#             opt_ce.step()
#
#     # Tiếp tục RL từ đây với lr thấp hơn
#     model = model_ce
#     optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
#     print('CE warmup done. Re-run training loop ở cell trên.')


In [ ]:
# Nếu restart kernel sau training, uncomment dòng dưới để load lại:
# model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'refresh_best.pt'), map_location=device))

refresh_scores = eval_rouge(model, test_data, test_loader, m)

results_df = pd.DataFrame({
    'Model':   ['LEAD', 'REFRESH'],
    'ROUGE-1': [lead_scores['ROUGE-1'], refresh_scores['ROUGE-1']],
    'ROUGE-2': [lead_scores['ROUGE-2'], refresh_scores['ROUGE-2']],
    'ROUGE-L': [lead_scores['ROUGE-L'], refresh_scores['ROUGE-L']],
}).set_index('Model')

print('── Kết quả thực nghiệm ──')
print(results_df.round(2).to_string())
print()
print('── Tham chiếu paper Table 2 (full CNN+DailyMail) ──')
print(pd.DataFrame({'LEAD': [39.6,17.7,36.2], 'REFRESH':[40.0,18.2,36.6]},
                   index=['ROUGE-1','ROUGE-2','ROUGE-L']).to_string())


In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('charts', exist_ok=True)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
metrics_list = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x, w = np.arange(3), 0.35
b1 = ax.bar(x - w/2, [lead_scores[kk]   for kk in metrics_list], w, label='LEAD', color='#5B7FA6')
b2 = ax.bar(x + w/2, [refresh_scores[kk] for kk in metrics_list], w, label='REFRESH', color='#E05C3A')
ax.set_xticks(x); ax.set_xticklabels(metrics_list); ax.set_ylabel('ROUGE F1 (%)')
ax.set_title(f'LEAD vs REFRESH ({len(test_data):,} test docs)'); ax.legend()
ax.bar_label(b1, fmt='%.2f', fontsize=9); ax.bar_label(b2, fmt='%.2f', fontsize=9)
plt.tight_layout(); plt.savefig('charts/chart_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(train_losses) + 1)
ax1.plot(ep, train_losses, 'o-', color='#E05C3A'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('RL Training Loss')
for key, ls in [('ROUGE-1','o-'),('ROUGE-2','s-'),('ROUGE-L','^-')]:
    ax2.plot(ep, [h[key] for h in val_rouge_history], ls, label=key)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('ROUGE F1 (%)'); ax2.set_title('Val ROUGE'); ax2.legend()
plt.tight_layout(); plt.savefig('charts/train_curves.png', dpi=150, bbox_inches='tight'); plt.show()

# Qualitative example
ex = test_data[0]
doc_t = torch.zeros(1, MAX_DOC_LEN, MAX_SENT_LEN, dtype=torch.long).to(device)
w_t   = torch.zeros(1, MAX_DOC_LEN).to(device)
for i, s in enumerate(ex['doc'][:MAX_DOC_LEN]):
    doc_t[0, i, :len(s)] = torch.tensor(s[:MAX_SENT_LEN], dtype=torch.long); w_t[0, i] = 1.0
model.eval()
with torch.no_grad():
    ref_idx = predict_topm(model(doc_t)[0], w_t, m)[0]

print(f'\n[GOLD]\n{ex["raw_gold"][:300]}...')
print('\n[LEAD]')
for i, s in enumerate(ex['raw_sents'][:m]): print(f'  [{i+1}] {" ".join(s)[:150]}')
print(f'\n[REFRESH] câu {ref_idx}')
for i in ref_idx:
    if i < len(ex['raw_sents']): print(f'  [{i+1}] {" ".join(ex["raw_sents"][i])[:150]}')

print('\nSaved: charts/chart_comparison.png, charts/train_curves.png')


In [ ]:
# ── In ĐIỂM của REFRESH cho từng câu (ví dụ minh hoạ — bài Palestine/ICC) ──
# predict_topm() có tính probs = P(extract) cho mỗi câu, nhưng chỉ giữ index top-m.
# Cell này tính lại probs và in ra điểm + xếp hạng để đưa lên slide.
ex = test_data[0]
doc_t = torch.zeros(1, MAX_DOC_LEN, MAX_SENT_LEN, dtype=torch.long).to(device)
w_t   = torch.zeros(1, MAX_DOC_LEN).to(device)
for i, s in enumerate(ex['doc'][:MAX_DOC_LEN]):
    doc_t[0, i, :len(s)] = torch.tensor(s[:MAX_SENT_LEN], dtype=torch.long)
    w_t[0, i] = 1.0

model.eval()
with torch.no_grad():
    logits, _ = model(doc_t)
    probs = F.softmax(logits.float(), dim=-1)[0, :, 0] * w_t[0]   # P(extract) cho mỗi câu

n_sent = len(ex['raw_sents'])
scores = probs[:n_sent].cpu().numpy()
order  = scores.argsort()[::-1]                                   # xếp hạng giảm dần theo điểm

print(f'Bài có {n_sent} câu — điểm P(extract) REFRESH gán cho từng câu:\n')
print(f'{"rank":<6}{"câu":<6}{"score":<10}text')
print('-' * 78)
for rank, idx in enumerate(order, 1):
    mark = '   <== CHON' if rank <= m else ''
    print(f'{rank:<6}{idx + 1:<6}{scores[idx]:<10.4f}{" ".join(ex["raw_sents"][idx])[:48]}{mark}')

# Bản theo thứ tự câu gốc (tiện đối chiếu câu 1/2/3 trên slide)
print('\nTheo thứ tự câu trong bài:')
for idx in range(min(n_sent, 8)):
    print(f'  câu {idx + 1}: score = {scores[idx]:.4f}')


## Hyperparameter Sweep — m, k, lr

In [ ]:
SWEEP_EPOCHS = 3
N_SWEEP = min(2000, N_TRAIN)
sweep_train, sweep_test = train_data[:N_SWEEP], test_data[:500]   # 500 docs (SEM ≈ 0.67 R1) — đủ phân biệt diff >1.3

def run_sweep(m_val, k_val, lr_val):
    if m_val == m and k_val == k:
        _train = sweep_train
    else:
        with Pool(processes=min(4, os.cpu_count())) as pool:
            oracles = list(pool.imap(_oracle_worker, [(item, m_val, p, k_val) for item in sweep_train], chunksize=100))
        _train = [{**item, 'oracles': o} for item, o in zip(sweep_train, oracles)]

    _model = REFRESHModel(
        ACTUAL_VOCAB_SIZE, EMB_DIM, SENT_EMB, LSTM_HID, NUM_KERNELS,
        pretrained_emb=embedding_matrix,   # ← Sweep models cũng init GloVe để fair comparison
    ).to(device)
    _opt   = torch.optim.Adam(_model.parameters(), lr=lr_val)
    dl_tr  = DataLoader(REFRESHDataset(_train, has_oracle=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    dl_te  = DataLoader(REFRESHDataset(sweep_test),              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    for _ in range(SWEEP_EPOCHS):
        _model.train()
        for batch in dl_tr:
            _opt.zero_grad()
            logits, _ = _model(batch['doc'].to(device))
            loss = reward_weighted_ce_loss(logits, batch['oracle_label'].to(device), batch['reward'].to(device), batch['weight'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(_model.parameters(), GRAD_CLIP); _opt.step()

    return eval_rouge(_model, sweep_test, dl_te, m_val)

sweep_results = []
for m_val in [2, 3, 4]:
    print(f'm={m_val}...', end=' ', flush=True)
    s = run_sweep(m_val, k, LR)
    sweep_results.append({'param':'m','value':m_val,**s}); print(f"R1={s['ROUGE-1']:.2f}")

for k_val in [3, 5, 10]:
    print(f'k={k_val}...', end=' ', flush=True)
    s = run_sweep(m, k_val, LR)
    sweep_results.append({'param':'k','value':k_val,**s}); print(f"R1={s['ROUGE-1']:.2f}")

for lr_val in [1e-3, 5e-4]:
    print(f'lr={lr_val}...', end=' ', flush=True)
    s = run_sweep(m, k, lr_val)
    sweep_results.append({'param':'lr','value':lr_val,**s}); print(f"R1={s['ROUGE-1']:.2f}")

sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (param, xlabel) in zip(axes, [('m','m (câu)'),('k','k (oracle)'),('lr','lr')]):
    sub = sweep_df[sweep_df['param'] == param]
    for key, ls in [('ROUGE-1','o-'),('ROUGE-2','s-'),('ROUGE-L','^-')]:
        ax.plot(sub['value'].astype(str), sub[key], ls, label=key)
    ax.set_xlabel(xlabel); ax.set_ylabel('ROUGE (%)'); ax.set_title(f'Sweep {param}'); ax.legend(fontsize=8)
plt.suptitle('Hyperparameter Sweep — REFRESH'); plt.tight_layout()
plt.savefig('charts/sweep_charts.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: charts/sweep_charts.png')


---
Copy `charts/*.png` → `../latex/images/` rồi compile slide Phần 5.

## EXPORT FOR APP -- bundle cho Streamlit demo
Chay **sau khi train xong**. Tao `refresh_app_bundle.zip` gom `refresh_best.pt` + `vocab.json` + `meta.json`. Tai ve, giai nen vao `app/model/` tren may local.


In [ ]:
# EXPORT FOR APP -- refresh_best.pt + vocab.json + meta.json -> zip -> download
import json, shutil

EXPORT_DIR = os.path.join(SAVE_DIR, 'app_bundle')
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copy(BEST_CKPT, os.path.join(EXPORT_DIR, 'refresh_best.pt'))            # 1. state_dict

with open(os.path.join(EXPORT_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:  # 2. vocab list (index=id)
    json.dump(vocab, f, ensure_ascii=False)

meta = {                                                                        # 3. config + ket qua thuc nghiem
    'emb_dim': EMB_DIM, 'lstm_hid': LSTM_HID, 'sent_emb': SENT_EMB,
    'num_kernels': NUM_KERNELS, 'max_doc_len': MAX_DOC_LEN, 'max_sent_len': MAX_SENT_LEN,
    'm': m, 'p': p, 'k': k, 'pad_id': PAD_ID, 'unk_id': UNK_ID, 'vocab_size': ACTUAL_VOCAB_SIZE,
    'faithful_rouge': FAITHFUL_ROUGE,
    'dataset': 'CNN/DailyMail 3.0.0',
    'embeddings': 'GloVe 200d (Wikipedia+Gigaword)',
    'n_train': len(train_data), 'n_val': len(val_data), 'n_test': len(test_data),
    'epochs': EPOCHS_RL, 'batch_size': BATCH_SIZE, 'lr': LR,
    'best_epoch': int(best_epoch), 'best_val_r1': round(float(best_val_r1), 2),
    'results': {
        'LEAD':    {kk: round(float(v), 2) for kk, v in lead_scores.items()},
        'REFRESH': {kk: round(float(v), 2) for kk, v in refresh_scores.items()},
    },
}
with open(os.path.join(EXPORT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

bundle = shutil.make_archive(os.path.join(SAVE_DIR, 'refresh_app_bundle'), 'zip', EXPORT_DIR)
print('Bundle :', bundle)
print('Gom    :', os.listdir(EXPORT_DIR))
print('meta   :', meta)
try:
    from google.colab import files
    files.download(bundle)        # auto tai ve khi chay tren Colab
except Exception:
    print('Khong phai Colab -- tu tai file zip o duong dan tren.')
